# `losses/`：Output-level KD Loss 手冊

## 1. 這個資料夾在整體架構中的位置

DataLoader 供應同一筆資料的 `teacher_input`、`reference_rgb` 和 `synthetic_target`。Burstormer Teacher 已凍結，只產生 `teacher_output`；NAFNet Student 產生 `student_output`。本資料夾的職責是分別量化 Student 與 clean target、Teacher output 的差異，供 training loop 組合 objective。

```text
teacher_input ──→ Burstormer (frozen) ──→ teacher_output ───────┐
                                                               ├─ OutputKDLoss → L_KD
reference_rgb ──→ NAFNet (trainable) ─→ student_output ─────────┤
clean target ──────────────────────────────────────────────────┴─ GroundTruthLoss → L_GT

L_total = L_GT + lambda_kd × L_KD → backward → optimizer 更新 Student
```

實作位於 `losses/distillation.py`：

- `GroundTruthLoss`：`L1(student_output, clean_reference)`。
- `OutputKDLoss`：`L1(student_output, teacher_output)`。

兩者都是 PyTorch `nn.Module`，輸入影像張量，輸出 scalar loss tensor。它們不載入模型、不產生資料、不決定 `lambda_kd`、不建立 optimizer、不負責 checkpoint 或 metrics。

## 2. 為什麼用兩項 loss？

**Supervised loss (`L_GT`)** 提供明確的 clean reference 監督，使 Student 不只模仿 Teacher，也被要求朝 clean target 學習。**KD loss (`L_KD`)** 讓 Student 的輸出接近 Teacher 的輸出，將多幀 Teacher 的輸出作為額外學習目標。

```text
L_total = L_GT + λ_KD · L_KD
```

`lambda_kd` 是總 loss 外部的超參數。數值愈大，Teacher output 在加權目標中的係數愈大；這不保證表現一定上升，也不能單靠公式推斷最佳值。現行 `train_kd.py` 預設 `0.1` 是啟動流程用的初始設定，尚未經正式實驗選定。Smoke runner 另有自己的設定，不能將兩者誤當作已調好的研究參數。

## 3. Teacher 為何不被更新？

Training loop 載入 Burstormer checkpoint 後，Teacher 進入 evaluation mode、所有參數關閉 gradient，Teacher forward 亦在 `torch.no_grad()` 之下執行。Optimizer 只接收 Student 的可訓練參數。故 `L_KD` 的 Teacher output 是 target，不會把梯度寫回 Teacher。

```text
Teacher params (requires_grad=False) → teacher_output (no grad)
Student params (requires_grad=True)  → student_output → L_GT / L_KD
                                                         ↓
                                                   optimizer.step()
```

`loss.item()` 只將 scalar 轉成 Python number 供顯示；反向傳播必須使用原本的 tensor，例如 `loss_total.backward()`。

## 4. 形狀與數值域契約

計算兩項 L1 loss 時，`student_output`、`teacher_output` 和 clean target 必須 shape 相同，正式 pipeline 預期 `[B, 3, H, W]`。目前 Teacher 對 batch size 1 有限制，故實際 `B=1`。

形狀相同並不足夠。三者也必須在同一數值域／顏色順序，並代表同一個 reference frame。若 Teacher output 做了 clamp 或其他 post-processing、但 target 或 Student output 沒有相同尺度，loss 數值就不能直接解讀。因此正式分數流程仍需確認線性化、white-level、gamma、clamp 與後處理規格。

## 5. 哪一段程式負責哪件事？

| 工作 | 所屬模組 |
|---|---|
| 定義 `L_GT`、`L_KD` | `losses/distillation.py` |
| 生成 noisy burst、sigma map、clean reference | `data/nafnet_kd_dataset.py` |
| Burstormer Teacher 載入／freeze | `models/burstormer/burstormer_teacher.py` |
| NAFNet Student 建構／初始化 | `models/nafnet/nafnet_student.py` |
| 設定 lambda、forward、loss 加權、backward、optimizer、validation 和保存權重 | `train_kd.py` |
| 假資料的一步 smoke | `utils/smoke_training.py`、`run_kd_smoke.py` |

Loss module 不需要對 Teacher「回傳」任何參數；Teacher checkpoint 不會被 loss 寫入。

## 6. 已實作與尚未實作

**已實作**：像素級 output L1 KD、clean-target L1 supervision；trainer 分別計算並印出 `loss_gt`、`loss_kd`、`loss_total`；teacher frozen，optimizer 更新 student。

**尚未實作**：feature-level distillation、perceptual loss、PSNR/SSIM/LPIPS evaluator、test split、post-processing。這些要另行設計及驗證，不能從目前 loss module 推斷已具備。

## 7. 執行與除錯

在專案根目錄可查看正式 trainer 參數：

```powershell
python train_kd.py --help
python train_kd.py --device cpu --epochs 1
```

若出現 shape mismatch，先比對 Dataset 的 `teacher_input`／`reference_rgb`／`synthetic_target` 和模型輸出 shape；若 loss 非 finite，檢查 noise data、模型 output、dtype 和 numeric range。若 KD loss 很小或很大，單一數值本身不能判斷模型優劣，需搭配 target range、驗證資料和正式 metrics。

Smoke loss 使用隨機 toy target，不可與訓練 run 的 validation loss 或任何 PSNR 結果直接比較。

## 8. `L1Loss` 的計算及解讀

目前兩個 class 都包裝 PyTorch `nn.L1Loss()`，預設 reduction 為 mean，亦即對所有 batch、channel、height、width 元素的絕對誤差取平均。它回傳單一 scalar，而不是 per-image/per-pixel map。

```text
L1(pred, target) = mean(abs(prediction - target))
```

使用 mean reduction 時，值會受 tensor 內元素平均差異影響，而不是直接反映圖片總像素數。即使 loss 是有限正數，也只代表數值可計算；若輸入 scale 不同、prediction/target 沒有對齊或 target 本身是 dummy tensor，數值不能代表有效去噪誤差。

## 9. Trainer 中的實際呼叫順序

`train_kd.py` 在每個 batch 先計算 Student output 與 Teacher output，再檢查它們和 clean target shape 是否一致。Training 時會清除 Student optimizer 的舊 gradients，算兩個 L1，依 `lambda_kd` 相加，檢查 total loss 是否 finite，最後 backward 與 optimizer step。Validation 時 Student/Teacher 都在 eval mode，且整段 validation 以 no-grad 執行，不更新參數。

```python
student_output = student(batch["reference_rgb"])
teacher_output = teacher(batch["teacher_input"])
loss_gt = ground_truth_loss(student_output, batch["synthetic_target"])
loss_kd = distillation_loss(student_output, teacher_output.detach())
loss_total = loss_gt + lambda_kd * loss_kd
loss_total.backward()
optimizer.step()
```

上面是訓練迴圈的概念順序；具體實作還包含 finite-value 和 shape checks、validation aggregation、checkpoint 儲存與錯誤處理。Loss classes 本身沒有這些副作用。

## 10. 為什麼記錄三種 Loss？

- `loss_gt` 反映 Student 相對 clean target 的監督項。
- `loss_kd` 反映 Student 和 frozen Teacher 輸出的平均 L1 差距。
- `loss_total` 是用於 backward 和 best-checkpoint 選擇的加權總和。

比較 train 與 validation loss 時，要確認 validation 使用 deterministic sample generation，且分割沒有 source image leakage。現行 validation Dataset 以固定 seed 重現每張圖片的 crop/noise；但 validation sample manifest 尚未單獨保存。`best_validation_loss` 用 validation `loss_total` 選 checkpoint，不等同用 test score 選模型。

Loss 曲線可以用來排查訓練是否發散、某項是否主導，但不能取代 test set 的影像品質指標。要宣稱 KD 帶來改善，還需與 supervised-only baseline 等其他必要對照在同一 test samples、相同 noise/metric protocol 下比較。

## 11. `lambda_kd` 的實驗意義與注意事項

`lambda_kd=0` 時，KD 項不影響 total objective，變成以 clean target 監督的 Student learning；`lambda_kd>0` 時 Teacher output 也影響 Student 更新。這只是 objective 的數學解釋，並不是建議立即把 0 當 baseline 或把任意非零值當最佳設定。

不同 loss 的 scale、Teacher output 是否已 clamp、Student output range、clean target 的值域都會影響實際梯度貢獻。故不能只看 lambda 數字大小判斷誰主導；需同時記錄 `loss_gt`、`lambda_kd * loss_kd`，必要時檢視 gradients 或執行受控的超參數實驗。現行 README/程式將 `0.1` 標為 placeholder/default，而非結論。

## 12. 未來擴充的邊界

如果之後考慮 feature-level distillation，需要定義兩個模型取出哪些層、如何對齊不同 channel count/spatial resolution、是否使用 projection adapter、特徵 loss 類型與權重。這些都不屬於目前 `OutputKDLoss`。切勿只在此 module 再加一個名為 `FeatureLoss` 的 class，卻未把 hooks/feature interface 和 training flow 完整接好。

若要增加 PSNR/SSIM，應放在 evaluation/metrics 模組，並明確指定 RGB 或 linear domain、data range、是否 crop border、是否對每張先算再平均、以及 post-process。Metrics 不應混進 training loss class，也不能用 training loss 值替代。

## 13. 常見錯誤及排查順序

| 現象 | 優先檢查 |
|---|---|
| `size mismatch` | Teacher / Student / clean target 是否同 shape，是否對齊同一 frame 和 crop |
| `loss.backward()` 說 tensor 沒 gradient | Student output 是否可微分、optimizer 是否包含 Student trainable parameters |
| Teacher parameter 出現 grad | Teacher 是否仍為 frozen/eval，forward 是否在 no-grad 執行 |
| loss NaN/Inf | Dataset inputs、noise parameters、model outputs、mixed precision/dtype 和 loss total 是否 finite |
| `loss_kd` 與 `loss_gt` 差距很大 | 先檢查兩種 target 的值域和尺度；不要未檢驗就改 lambda 掩蓋 |
| validation loss 低但視覺結果差 | 驗證 sample 數、資料 split、loss range；正式結論要有 test metrics 和圖像檢視 |

偵錯時應記錄 shape、min/max、mean、finite status 和 sample metadata。避免直接 clamp output 或修改 target，除非該處理是明確 protocol 並同時套用在對照組。
